In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="font-family:Arial, sans-serif; font-size:16px; line-height:1.40; width:1050px;">

<div style="font-size:22px; font-weight:bold; color:#12388c; margin-bottom:8px;">
Correlated Noise from an Independent Bernoulli Sequence
</div>

<div style="margin-bottom:4px;">
<b>Input:</b> w[n] is an independent Bernoulli sequence.
</div>

<div style="margin-bottom:4px;">
<b>System:</b> x[n] = αx[n−1] + w[n], with |α| &lt; 1.
</div>

<div style="margin-bottom:4px;">
The feedback term αx[n−1] introduces memory and therefore correlation between output samples.
</div>

<div>
<b>This notebook:</b> compares the independent input sequence with the correlated output and its autocorrelation.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='160px')

alpha_slider = FloatSlider(min=0.0, max=0.95, step=0.05, value=0.75, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

p_slider = FloatSlider(min=0.10, max=0.90, step=0.05, value=0.50, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

N_slider = IntSlider(min=200, max=2000, step=100, value=800, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

lag_slider = IntSlider(min=10, max=80, step=5, value=30, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

alpha_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.75</div>')

p_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.50</div>')

N_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">800</div>')

lag_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">30</div>')

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_alpha_value(change):
    alpha_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{alpha_slider.value:.2f}</div>'

def update_p_value(change):
    p_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{p_slider.value:.2f}</div>'

def update_N_value(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

def update_lag_value(change):
    lag_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{lag_slider.value}</div>'

alpha_slider.observe(update_alpha_value, names='value')
p_slider.observe(update_p_value, names='value')
N_slider.observe(update_N_value, names='value')
lag_slider.observe(update_lag_value, names='value')

# ============================================================
# CONTROL LABELS
# ============================================================

alpha_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Feedback α:</div>')

p_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Bernoulli p:</div>')

N_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Samples N:</div>')

lag_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Maximum lag:</div>')

# ============================================================
# CONTROLS GRID
#
# Row 1: Feedback alpha | Bernoulli p
# Row 2: Samples N      | Maximum lag
# ============================================================

controls_grid = GridBox(
    children=[
        alpha_label, alpha_slider, alpha_value,
        p_label, p_slider, p_value,
        N_label, N_slider, N_value,
        lag_label, lag_slider, lag_value
    ],
    layout=Layout(
        width='790px',
        grid_template_columns='105px 160px 55px 125px 160px 55px',
        grid_template_rows='34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="font-family:Arial; font-size:17px; font-weight:bold; color:#12388c; margin-bottom:5px;">
        Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='820px',
        padding='10px 14px',
        border='1px solid #d2d2d2',
        overflow='hidden',
        margin='10px 0px 10px 0px'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_correlated_noise(alpha=0.75, p=0.50, N=800, max_lag=30):

    # --------------------------------------------------------
    # INDEPENDENT BERNOULLI INPUT
    # --------------------------------------------------------

    rng = np.random.default_rng(42)

    w = rng.binomial(1, p, N).astype(float)

    # --------------------------------------------------------
    # CORRELATED OUTPUT
    #
    # x[n] = alpha*x[n-1] + w[n]
    # --------------------------------------------------------

    x = np.zeros(N)

    x[0] = w[0]

    for n in range(1, N):
        x[n] = alpha * x[n - 1] + w[n]

    # --------------------------------------------------------
    # REMOVE INITIAL TRANSIENT
    # --------------------------------------------------------

    transient = min(100, N // 10)

    w_ss = w[transient:]

    x_ss = x[transient:]

    # --------------------------------------------------------
    # CENTERED SEQUENCES
    # --------------------------------------------------------

    w_centered = w_ss - np.mean(w_ss)

    x_centered = x_ss - np.mean(x_ss)

    # ========================================================
    # INPUT AUTOCORRELATION
    # ========================================================

    full_w_corr = np.correlate(w_centered, w_centered, mode='full')

    center_w = len(full_w_corr) // 2

    lags = np.arange(-max_lag, max_lag + 1)

    Rww = full_w_corr[center_w - max_lag:center_w + max_lag + 1]

    normalization_w = len(w_centered) - np.abs(lags)

    Rww = Rww / normalization_w

    if Rww[max_lag] != 0:
        Rww = Rww / Rww[max_lag]

    # ========================================================
    # OUTPUT AUTOCORRELATION
    # ========================================================

    full_x_corr = np.correlate(x_centered, x_centered, mode='full')

    center_x = len(full_x_corr) // 2

    Rxx = full_x_corr[center_x - max_lag:center_x + max_lag + 1]

    normalization_x = len(x_centered) - np.abs(lags)

    Rxx = Rxx / normalization_x

    if Rxx[max_lag] != 0:
        Rxx = Rxx / Rxx[max_lag]

    # ========================================================
    # THEORETICAL NORMALIZED OUTPUT AUTOCORRELATION
    #
    # rho_x[k] = alpha^|k|
    # ========================================================

    Rxx_theory = alpha ** np.abs(lags)

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(10.4, 6.8))

    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.05], hspace=0.46, wspace=0.30)

    ax1 = fig.add_subplot(gs[0, 0])

    ax2 = fig.add_subplot(gs[0, 1])

    ax3 = fig.add_subplot(gs[1, :])

    # ========================================================
    # GRAPH 1: BERNOULLI INPUT
    # ========================================================

    show_N = min(200, len(w_ss))

    ax1.step(np.arange(show_N), w_ss[:show_N], where='mid', linewidth=1.1)

    ax1.set_xlim(0, show_N - 1)

    ax1.set_ylim(-0.2, 1.2)

    ax1.set_xlabel('Time index n', fontsize=11)

    ax1.set_ylabel('w[n]', fontsize=11)

    ax1.set_title('Independent Bernoulli Input', fontsize=13, pad=9)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 2: CORRELATED OUTPUT
    # ========================================================

    ax2.plot(np.arange(show_N), x_ss[:show_N], linewidth=1.1)

    ax2.set_xlim(0, show_N - 1)

    ax2.set_xlabel('Time index n', fontsize=11)

    ax2.set_ylabel('x[n]', fontsize=11)

    ax2.set_title(f'Correlated Output, α = {alpha:.2f}', fontsize=13, pad=9)

    ax2.tick_params(axis='both', labelsize=9)

    ax2.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 3: AUTOCORRELATION
    # ========================================================

    ax3.plot(lags, Rww, linewidth=1.5, label='Bernoulli input autocorrelation')

    ax3.plot(lags, Rxx, linewidth=1.8, label='Estimated output autocorrelation')

    ax3.plot(lags, Rxx_theory, linestyle='--', linewidth=2.0, label='Theoretical output autocorrelation')

    ax3.axhline(0, linewidth=0.8)

    ax3.axvline(0, linewidth=0.8, linestyle=':')

    ax3.set_xlim(-max_lag, max_lag)

    ax3.set_ylim(-0.25, 1.15)

    ax3.set_xlabel('Lag k', fontsize=11)

    ax3.set_ylabel('Normalized autocorrelation', fontsize=11)

    ax3.set_title('Input and Output Autocorrelation', fontsize=13, pad=9)

    ax3.tick_params(axis='both', labelsize=9)

    ax3.grid(True, linestyle=':', alpha=0.5)

    ax3.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=8)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(left=0.08, right=0.97, top=0.92, bottom=0.14)

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL RESULTS
    # ========================================================

    theoretical_mean = p / (1.0 - alpha)

    theoretical_variance = p * (1.0 - p) / (1.0 - alpha**2)

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:900px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Steady-state theoretical mean:</b>
    E{{x[n]}} = p/(1−α) = {theoretical_mean:.4f}

    &nbsp;&nbsp;&nbsp;

    <b>Estimated mean:</b>
    {np.mean(x_ss):.4f}

    <br>

    <b>Steady-state theoretical variance:</b>
    p(1−p)/(1−α²) = {theoretical_variance:.4f}

    &nbsp;&nbsp;&nbsp;

    <b>Estimated variance:</b>
    {np.var(x_ss):.4f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_correlated_noise,
    {
        'alpha': alpha_slider,
        'p': p_slider,
        'N': N_slider,
        'max_lag': lag_slider
    }
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1050px;
    padding:11px 15px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="font-size:18px; font-weight:bold; color:#197b35; margin-bottom:6px;">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The Bernoulli input samples are independent, so their centered autocorrelation is concentrated near lag k = 0.
</div>

<div style="margin-bottom:4px;">
The feedback term αx[n−1] introduces memory into the output, producing nonzero correlation for k ≠ 0.
</div>

<div>
As α increases toward 1, the output autocorrelation decays more slowly and the process retains memory for a longer time.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)